# SCPN Control: Full-Stack Physics & Logic Demo (2026)

This notebook demonstrates the vertically integrated SCPN Control framework, showcasing the high-performance Rust math kernels, the Neuro-Symbolic SNN controller, and the multi-layer Kuramoto phase dynamics engine.

## 1. System Initialization
Verify that the unified Rust backend is active and the environment is correctly configured.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scpn_control_rs as rs
from scpn_control.core import FusionKernel, RUST_BACKEND
from scpn_control.scpn import NeuroSymbolicController, FusionCompiler, StochasticPetriNet
from scpn_control.phase import UPDESystem, KnmSpec, build_knm_plasma, plasma_omega

print(f"SCPN Control Backend: {'RUST (High Performance)' if RUST_BACKEND else 'NumPy (Fallback)'}")
print(f"Rust Native Module: {rs.__version__ if hasattr(rs, '__version__') else 'Verified'}")

## 2. Grad-Shafranov Equilibrium
We use the **Rust Multigrid Solver** to compute a high-fidelity 2D magnetic equilibrium for an ITER-like configuration. The solver now correctly implements $P'$ and $FF'$ profiles using the **mTanh** formulation.

In [ ]:
# Initialize kernel from config
fk = FusionKernel("../iter_config.json")

# Solve equilibrium (Picard + Multigrid)
print("Solving Grad-Shafranov equilibrium...")
res = fk.solve_equilibrium()

print(f"Converged:  {res['converged']}")
print(f"Iterations: {res['iterations']}")
print(f"Solve Time: {res['solve_time_ms']:.2f} ms")

# Plot poloidal flux surfaces
psi = fk.state.psi
plt.figure(figsize=(8, 10))
plt.contour(fk.grid.r, fk.grid.z, psi, levels=30)
plt.title("Poloidal Flux Surfaces ($\psi$) — Rust Multigrid")
plt.xlabel("R [m]")
plt.ylabel("Z [m]")
plt.axis("equal")
plt.show()

## 3. Neuro-Symbolic Logic (SNN)
We compile a **Stochastic Petri Net** into a Spiking Neural Network (SNN) artifact. The controller executes in the Rust fast-path, achieving >13 kHz throughput.

In [ ]:
# 1. Define Petri Net logic
net = StochasticPetriNet()
net.add_place("x_R_pos", initial_tokens=0.5)
net.add_place("x_R_neg", initial_tokens=0.0)
net.add_transition("t_PF", threshold=0.2)
net.add_arc("x_R_pos", "t_PF", weight=1.0)

# 2. Compile to SNN Artifact
compiler = FusionCompiler()
compiled_net = compiler.compile(net)
artifact = compiled_net.export_artifact(
    name="demo_artifact",
    readout_config={
        "action_names": ["dI_PF_A"],
        "pos_places": [net._place_idx["x_R_pos"]],
        "neg_places": [net._place_idx["x_R_neg"]],
        "gains": [1.0], "abs_max": [10.0], "slew_per_s": [100.0]
    },
    injection_config=[
        {"place_id": net._place_idx["x_R_pos"], "source": "R_axis_err", "scale": 1.0, "offset": 0.0}
    ]
)

# 3. Run Controller
controller = NeuroSymbolicController(artifact, seed_base=42)
obs = {"R_axis_m": 6.1}  # Plant measurement
action = controller.step(obs, k=0)

print(f"Controller Action: {action}")
print(f"Active Backend:   {controller.runtime_backend_name}")

## 4. Multi-Layer Phase Dynamics (16-Layer Kuramoto)
The **Unified Phase Dynamics Equation (UPDE)** simulates multi-scale synchronization across the 16 plasma layers defined in Paper 27. The integration uses the **Rust `upde_tick` fast-path**.

In [ ]:
# Build 16-layer plasma hierarchy
L = 16
spec = build_knm_plasma(mode="hybrid", L=L)
upde = UPDESystem(spec, dt=1e-3, psi_mode="global_mean_field")

# Initial phases and natural frequencies
n_per = 50
theta_init = [np.random.uniform(0, 2*np.pi, n_per) for _ in range(L)]
omega_val = plasma_omega(L)
omega_layers = [np.full(n_per, omega_val[m]) for m in range(L)]

# Run simulation
print(f"Simulating {L}-layer synchronization manifold...")
out = upde.run(n_steps=500, theta_layers=theta_init, omega_layers=omega_layers)

# Plot Synchronization (Order Parameter R)
plt.figure(figsize=(10, 5))
plt.plot(out["R_global_hist"], 'k-', lw=2, label="Global Coherence (R)")
for m in [0, 7, 15]:
    plt.plot(out["R_layer_hist"][:, m], alpha=0.6, label=f"Layer {m}: {spec.layer_names[m]}")
plt.title("16-Layer Plasma Synchronization (Paper 27 Hierarchy)")
plt.xlabel("Time Step")
plt.ylabel("Order Parameter R")
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

## 5. Performance Summary
The vertical integration of Rust into the SCPN stack provides the following verified benchmarks on this machine.

In [ ]:
import time

print("Verifying loop throughput...")
n_bench = 1000
t0 = time.perf_counter()
for k in range(n_bench):
    _ = controller.step({"R_axis_m": 6.2 + 0.1*np.sin(k*0.1)}, k)
t1 = time.perf_counter()

avg_us = (t1-t0)/n_bench * 1e6
print(f"Control Step Latency: {avg_us:.2f} us")
print(f"Control Throughput:    {1e6/avg_us:.1f} Hz")